In [1]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

#test and make sure the drive is properly mounted
!ls "/content/drive/Shareddrives/EMG POSE deep learning/Erdos_dl_emg_pose/" # this should list all files in the shared drive

Mounted at /content/drive
checkpoints   emg2pose	     model.py		scaler		     welford.py
CNN_model.py  EMG_input.png  __pycache__	training_curves.png
data	      load_data.py   requirements.gdoc	train.py


In [2]:
from google.colab import auth
auth.authenticate_user()
from google.auth import default
import gspread

creds, _ = default()
gc = gspread.authorize(creds)

import os
import sys

sys.path.append("/content/drive/Shareddrives/EMG POSE deep learning/Erdos_dl_emg_pose/")

import pandas as pd

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, ConcatDataset

from data.session import Emg2PoseSessionData, WindowedEmgDataset
from data.transforms import ExtractToTensor, RotationAugmentation, Compose
from data.utils import load_splits, downsample, get_ik_failures_mask
from data.alignment import align_predictions, align_mask
from load_data import build_datasets, build_dataloaders, demonstrate_alignment

In [5]:
# ---------------------------------------------------------------------------
# Constants from emg2pose
# ---------------------------------------------------------------------------
EMG_SAMPLE_RATE = 2000      # Hz
EMG_CHANNELS = 16
JOINT_ANGLE_CHANNELS = 20
WINDOW_LENGTH = 2000        # 1 second at 2kHz (training)
VAL_WINDOW_LENGTH = 10000   # 5 seconds at 2kHz (validation/test)
BATCH_SIZE = 64 #if total number of files is less than 30, 8 will be selected by default

# Default data location (mini dataset in sibling repo)
DEFAULT_DATA_DIR = Path("/content/drive/Shareddrives/EMG POSE deep learning/data/data_subset").resolve()
print(DEFAULT_DATA_DIR)


/content/drive/Shareddrives/EMG POSE deep learning/data/data_subset


In [6]:
# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main(data_dir, metadata, batch_size, num_workers):

	# --- Real dataset mode ---
	data_dir = Path(data_dir).expanduser() if data_dir else DEFAULT_DATA_DIR
	assert data_dir.exists(), f"Data directory not found: {data_dir}"

	print(f"=== Loading data from {data_dir} ===")

	# Load splits from CSV and filter to files that exist on disk
	# (the full metadata CSV has 25k sessions; mini dataset has 30)
	splits = load_splits(metadata)
	available = {p.stem for p in data_dir.glob("*.hdf5")}

	for split_name in splits:
		before = len(splits[split_name])
		splits[split_name] = [f for f in splits[split_name] if f in available]
		after = len(splits[split_name])
		if after < before:
			print(f"  {split_name}: {before} in CSV -> {after} on disk")

	# If any split is empty after filtering (e.g. mini dataset where all
	# files map to one split), redistribute available files 70/15/15
	has_empty = any(len(splits[s]) == 0 for s in ["train", "val", "test"])
	if has_empty:
		all_files = sorted(available)
		np.random.seed(42)
		np.random.shuffle(all_files)
		n = len(all_files)
		n_train = max(1, int(0.7* n))
		n_val = max(1, int(0.15 * n))
		splits = {
			"train": all_files[:n_train],
			"val": all_files[n_train:n_train + n_val],
			"test": all_files[n_train + n_val:],
		}
		print(f"\n  Redistributed {n} files into 70/15/15 splits")

	def resolve_paths(filenames):
		return [data_dir / f"{name}.hdf5" for name in filenames]

	print(f"\nUsing splits: train={len(splits['train'])}, "
			f"val={len(splits['val'])}, test={len(splits['test'])}")

	# With very few files, use smaller batch size
	n_total = sum(len(v) for v in splits.values())
	batch_size = min(batch_size, 8) if n_total <= 30 else batch_size

	loaders = build_dataloaders(
		train_paths=resolve_paths(splits["train"]),
		val_paths=resolve_paths(splits["val"]),
		test_paths=resolve_paths(splits["test"]),
		batch_size=batch_size,
		num_workers=num_workers,
	)

	# --- Print data summary ---
	print(f"\n{'='*50}")
	print(f"{'Split':<8} {'Batches':>8} {'Samples':>10}")
	print(f"{'='*50}")
	for name, loader in loaders.items():
		n_samples = len(loader.dataset)
		n_batches = len(loader)
		print(f"{name:<8} {n_batches:>8} {n_samples:>10}")

	# --- Fetch one batch and inspect ---
	train_loader = loaders["train"]
	batch = next(iter(train_loader))

	print(f"\n--- First training batch ---")
	for key, val in batch.items():
		if isinstance(val, torch.Tensor):
			print(f"  {key:<20} {str(val.shape):<25} dtype={val.dtype}")
		else:
			print(f"  {key:<20} {val}")

	# --- IK failure mask summary ---
	mask = batch["no_ik_failure"]
	pct_valid = 100 * mask.float().mean().item()
	print(f"\n  IK failure mask: {pct_valid:.1f}% valid samples in batch")

	# --- Downsampling demo ---
	emg_np = batch["emg"][0].numpy()  # (C, T) for first sample
	emg_ds = downsample(emg_np.T, native_fs=EMG_SAMPLE_RATE, target_fs=30)
	print(f"\n--- Downsampling Demo ---")
	print(f"  Original:     {emg_np.shape[1]} samples at {EMG_SAMPLE_RATE} Hz")
	print(f"  Downsampled:  {emg_ds.shape[0]} samples at 30 Hz")

	# --- Alignment demo ---
	demonstrate_alignment(batch)

	return batch

worksheet = gc.open('metadata').sheet1

rows = worksheet.get_all_values()
columns = ['session' ,'user', 'stage','start', 'end', 'side' ,
					'filename' ,'moving_hand', 'held_out_user' ,'held_out_stage',
					'split' ,'generalization']
metadata = pd.DataFrame.from_records(rows, columns=columns)
metadata = metadata.drop(index=metadata.index[0])

batch = main(data_dir=DEFAULT_DATA_DIR,
     		metadata = metadata,
		 		batch_size=BATCH_SIZE,
		 		num_workers=0)

SpreadsheetNotFound: <Response [200]>